In [ ]:
# Install hypertools (dev-1.0 preview) -- run this first on Colab.
# On release this becomes: %pip install hypertools
%pip install -q "hypertools[interactive] @ git+https://github.com/ContextLab/hypertools.git@dev-1.0"

%matplotlib inline

# A market as one moving path: chemtrails + a live forecast

This tutorial folds several daily financial series into a single moving 3-D **market path**, colors it by an equal-weight index with a labeled colorbar, and animates it with **chemtrails** while the camera makes **one slow quarter-turn** over the clip (`rotations=0.25` -- enough parallax to read the 3-D shape without the box spinning out from under the overlay). On top of the library call we overlay a **live Kalman forecast**.

The forecast is anchored at the **true on-screen head** of the plotted line (read from `ani._args[1][0]` each frame), not at raw reduce-space coordinates -- `hyp.plot` normalizes the path into its drawn cube, so reduce-space points don't line up with what's on screen. We fit the (reduce -> drawn) scale so the forecast delta lands in drawn units, clip it inside the box, and let every *past* forecast linger as a faint-blue "history of predictions" fan behind the bright-red current one.

A subtitle under the title keeps a running **directional-accuracy** score: every forecast is compared with what the path actually did over the same horizon, and only counts once that horizon has elapsed on screen (no peeking ahead). 50% would be a coin flip.

Data comes from [FRED](https://fred.stlouisfed.org) as small CSVs (cached on disk); if the network is unavailable we fall back to a synthetic basket of correlated random-walk assets, so the notebook always runs.

## 1. Imports and a disk cache

In [2]:
import os, tempfile, urllib.request
import numpy as np
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
import hypertools as hyp

CACHE = os.path.join(tempfile.gettempdir(), 'hypertools_tutorial')
os.makedirs(CACHE, exist_ok=True)

## 2. Fetch the data (with a synthetic fallback)

We pull five broad daily series from FRED. If anything goes wrong (network, parsing), `fetch_fred` returns `None` and we synthesize a basket of correlated assets instead.

In [3]:
FRED_IDS = ['SP500', 'NASDAQCOM', 'DGS10', 'DCOILWTICO', 'VIXCLS']
START, END = '2004-01-01', '2024-01-01'


def fetch_fred(ids, start, end):
    try:
        import pandas as pd
        frames = []
        for sid in ids:
            url = ('https://fred.stlouisfed.org/graph/fredgraph.csv'
                   f'?id={sid}&cosd={start}&coed={end}')
            dest = os.path.join(CACHE, f'fred_{sid}.csv')
            if not (os.path.exists(dest) and os.path.getsize(dest)):
                req = urllib.request.Request(
                    url, headers={'User-Agent': 'ht-tutorial/1.0'})
                with urllib.request.urlopen(req, timeout=30) as r:
                    open(dest, 'wb').write(r.read())
            df = pd.read_csv(dest, na_values=['.'])
            df.columns = ['date', sid]
            df['date'] = pd.to_datetime(df['date'])
            frames.append(df.set_index('date'))
        # sort=False + an explicit sort_index(): pandas is deprecating
        # the implicit sort, and we sort the union index ourselves
        merged = pd.concat(frames, axis=1,
                           sort=False).sort_index().ffill().dropna()
        return merged.to_numpy(dtype=float)
    except Exception:
        return None


def synthetic_basket(n_days=1000, n_assets=5, seed=0):
    rng = np.random.default_rng(seed)
    market = np.cumsum(rng.standard_normal(n_days)) * 0.6
    cols = []
    for k in range(n_assets):
        idio = np.cumsum(rng.standard_normal(n_days)) * (0.4 + 0.1 * k)
        cols.append(100 * np.exp(0.02 * (market + idio) / 10))
    return np.column_stack(cols)


prices = fetch_fred(FRED_IDS, START, END)
if prices is None:
    prices = synthetic_basket()
print(prices.shape)

(1941, 5)


## 3. Reduce to a single smoothed 3-D path

We thin to a manageable number of frames (keeping the thinning factor `THIN`, which the anchor spacing below is derived from), take the **log** prices (assets grow ~exponentially; log linearizes the path), and hand them to a single `hyp.reduce` call. We also build an equal-weight index to color the path.

`hyp.reduce(logp, reduce='IncrementalPCA', ndims=3, manip='Smooth', normalize='across')` runs hypertools' canonical stage order, `manip -> normalize -> reduce`:

* **`manip='Smooth'`** applies a Savitzky-Golay filter per dataset *before* the reduction, so day-to-day jitter is removed and the path reads as a curve instead of a scribble.
* **`normalize='across'`** z-scores each column across the stacked rows, putting five series measured on very different scales (an index level, a yield, a volatility quote) into comparable units. Passing it here replaces a hand-rolled z-score.
* **`reduce='IncrementalPCA'`** projects those z-scored series onto their 3 highest-variance directions in mini-batches, so each day becomes one 3-D point.
* **`ndims=3`** is what makes the result plottable: the 3 columns are exactly the x/y/z of the drawn path.

In [4]:
THIN = max(1, len(prices) // 800)                         # keep frames manageable
prices = prices[::THIN]
idx_level = (prices / prices[0]).mean(axis=1) * 100.0     # equal-weight index
logp = np.log(np.clip(prices, 1e-9, None))                # log linearizes growth
# ONE call, in hypertools' canonical stage order (manip -> normalize ->
# reduce): Smooth strips day-to-day jitter, normalize='across' z-scores the
# columns across the stacked rows (replacing a hand-rolled z-score), and
# IncrementalPCA keeps the 3 highest-variance directions.
red = np.asarray(hyp.reduce(logp, reduce='IncrementalPCA', ndims=3,
                            manip='Smooth', normalize='across'))
T = len(red)

## 4. Precompute Kalman forecasts at monthly anchors -- and score them

Running a Kalman filter on the full daily path at every frame would be too slow, so we forecast at monthly anchors from the history-so-far. Each forecast is stored as a **reduce-space delta**; later we convert those deltas into drawn units and hang them off the true on-screen head.

`hyp.predict(hist, model='Kalman', t=HORIZON)` fits a linear-Gaussian state-space model to the history and rolls it forward, returning exactly `t` **new** rows in the same reduced space `hist` lives in. All `t` rows are future steps: the first returned row is the *first forecast step*, not a copy of the last observation. The displacement therefore has to be anchored on the last observed row, `f - hist[-1]`. Anchoring on `f - f[0]` instead is a bug: it throws away a whole step of the forecast and forces the first displacement to zero. We prepend an explicit zero row so the drawn line still starts exactly at the current head.

Two details matter for forecast *quality*:

* **`MIN_HIST`** -- we refuse to forecast until there is a real run-up of history. Kalman fits from only a couple of samples are wildly over-confident: measured on the FRED series above, dropping this requirement lets the largest forecast reach ~25x the median forecast length, and the visual gain we apply later turns that into an arrow that streaks across the box. Requiring two years of monthly history halves the worst case (to ~12x the median) without changing directional skill.
* **scoring** -- each forecast is compared with the path's *actual* displacement over the same horizon. A "hit" is a positive dot product: the forecast pointed the right way. This is what the on-screen accuracy readout reports.

**Which forecaster?** Measured rather than assumed. On a 20-stock daily version of this pipeline (293 anchors, scoring the predicted 4-month displacement against what the market actually did), direction was called correctly by Kalman 51%, ARIMA 51%, and `Laplace` 65%, against 62% for the trivial "assume it keeps drifting the way it has been" rule. A reduced market path is close to a random walk with drift, so a single linear-Gaussian fit has little to grip on; hypertools' `Laplace` is a likelihood-weighted Bayesian ensemble over a whole population of candidate forecasters and holds up better. Kalman is used here only because it is roughly 30x faster; swap one keyword to `model='Laplace'` for the better forecast.

Read those numbers with care. The 293 horizons overlap, so the effective sample is nearer 70, and 65% against a 62% baseline is well inside the noise. Most of the skill on offer is the market's long upward drift, not the model reading the current path.

In [5]:
# STEP is derived from the thinning factor, so one anchor step stays ~1 trading
# month of the ORIGINAL series no matter how much we thinned.
STEP = max(2, round(21 / THIN))                           # ~1 trading month
HORIZON = 4                                               # months ahead
MIN_HIST = 24                                             # monthly samples
anchors = list(range(MIN_HIST * STEP, T, STEP))
raw_fc, raw_hit = {}, {}
for a in anchors:
    hist = red[:a + 1:STEP]
    if len(hist) < 2:
        continue
    f = np.asarray(hyp.predict(hist, model='Kalman', t=HORIZON))
    # the `t` returned rows are ALL future steps, so f[0] is the FIRST
    # forecast step, not the last observation: anchor the displacement on the
    # last OBSERVED row. (`f - f[0]` would discard a whole step and force the
    # first displacement to zero.) The prepended zero row keeps the drawn
    # forecast starting exactly at the current head.
    raw_fc[a] = np.vstack([np.zeros((1, f.shape[1])),
                           f - hist[-1]])                 # reduce-space delta
    # score it against what ACTUALLY happened over the same horizon: did the
    # forecast point the right way? (directional hit = positive dot product)
    j = min(T - 1, a + HORIZON * STEP)
    d_pred, d_act = raw_fc[a][-1], red[j] - red[a]
    n1, n2 = np.linalg.norm(d_pred), np.linalg.norm(d_act)
    raw_hit[a] = (float(d_pred @ d_act) > 0) if (n1 > 1e-12 and n2 > 1e-12) \
        else None
print(f'{len(raw_fc)} forecasts from {MIN_HIST}+ monthly samples of history')

74 forecasts from 24+ monthly samples of history


## 5. Plot with a slow quarter-turn, then overlay the head-anchored forecast

The library call itself is short. `reduce=None` switches off `hyp.plot`'s default IncrementalPCA, because `red` is already 3-D from our own reduction; `hue=idx_level` is one continuous value per point, mapped to colors through `palette='plasma'` (`colorbar=False` because we draw our own labeled colorbar); and `chemtrails=True` keeps the whole path traversed so far faintly visible behind the moving head.

We plot the path with `chemtrails=True` and `rotations=0.25` (one slow quarter-turn over the whole clip). `duration` and `frame_rate` **must** be passed: otherwise `hyp.plot` falls back to its 30s/30fps defaults while our own `total` says something else, and the forecasts desync from the frames.

Then we read the **true drawn head** from the plotted line artist (`ani._args[1][0]`), fit the per-axis (reduce -> drawn) scale by revealing the path fully once and comparing it to the reduce-space coordinates, and convert each stored forecast delta into drawn units. A single visual `GAIN` makes the median forecast a legible fraction of the box, and a `CAP` at **1.8x the median length** keeps the heavy tail of Kalman magnitudes from streaking across it -- relative differences still read (a bigger predicted move is a longer arrow), which an absolute cap would flatten.

The legend is built from **real `Line2D` handles**, so each entry is drawn in exactly the style it has on screen, and the running accuracy gets its own subtitle line under the title.

In [6]:
# THE hypertools call: one market path, hue = equal-weight index, chemtrails,
# and ONE slow quarter-turn of the camera (rotations=0.25) over the clip.
# duration/frame_rate MUST be passed (see above) or the frames desync.
duration, fps = 8, 20
fig, ani = hyp.plot(red, fmt='-', reduce=None, hue=idx_level, colorbar=False,
                    palette='plasma', animate=True, chemtrails=True,
                    rotations=0.25, duration=duration, frame_rate=fps,
                    linewidth=2.2, size=(9, 6.5), show=False)
ax = [a for a in fig.axes if hasattr(a, 'zaxis')][0]
ax.set_position([0.0, 0.03, 0.78, 0.9])
total = int(round(fps * duration))

# read the visible line artist so forecasts anchor at the TRUE drawn head, and
# fit the (reduce -> drawn) per-axis scale so the reduce-space delta lands in
# drawn units
market_line = ani._args[1][0]
_orig = ani._func
_orig(total - 1, *ani._args)                              # reveal fully, once
_fx, _fy, _fz = market_line.get_data_3d()
full_drawn = np.column_stack([_fx, _fy, _fz])
K = len(full_drawn)
_red_rs = np.column_stack([np.interp(np.linspace(0, T - 1, K), np.arange(T),
                                     red[:, c]) for c in range(3)])
SLOPE = np.array([np.polyfit(_red_rs[:, c], full_drawn[:, c], 1)[0]
                  for c in range(3)])
BLO = np.array([ax.get_xlim3d()[0], ax.get_ylim3d()[0], ax.get_zlim3d()[0]])
BHI = np.array([ax.get_xlim3d()[1], ax.get_ylim3d()[1], ax.get_zlim3d()[1]])


def _frame_of(a):
    """Frame at which reduce-space sample `a` is the animated head."""
    return int(round(a / max(1, T - 1) * (total - 1)))


# forecasts as DRAWN-space deltas, keyed by the frame they were made; a single
# GAIN makes them legible (hyp packs this path into a sub-region of the cube),
# and the CAP at 1.8x the MEDIAN length tames the heavy tail without flattening
# every large forecast onto one length (as an absolute cap would)
FC = {_frame_of(a): SLOPE[None, :] * d for a, d in raw_fc.items()}
_ends = [np.linalg.norm(d[-1]) for d in FC.values() if len(d)]
MED_LEN = 0.20                                            # median arrow length
GAIN = MED_LEN / (np.median(_ends) or 1.0)
CAP = 1.8 * MED_LEN


def _scale(d):
    d = GAIN * d
    L = np.linalg.norm(d[-1])
    return d * (CAP / L) if L > CAP else d


FC = {f: _scale(d) for f, d in FC.items()}
frame_list = sorted(FC)
HEAD_CACHE = {}

# running directional accuracy: a forecast only counts once its horizon has
# actually elapsed on screen (no peeking at the future)
_matured = sorted((_frame_of(min(T - 1, a + HORIZON * STEP)), raw_hit[a])
                  for a in raw_fc if raw_hit.get(a) is not None)
ACC = np.full(total, np.nan)                              # frame -> accuracy %
_n = _k = _mi = 0
for _f in range(total):
    while _mi < len(_matured) and _matured[_mi][0] <= _f:
        _n += 1
        _k += int(_matured[_mi][1])
        _mi += 1
    if _n:
        ACC[_f] = 100.0 * _k / _n
N_SCORED = _n
print(f'forecasts: {len(FC)} drawn, {N_SCORED} scored; '
      f'final directional accuracy = {ACC[-1]:.0f}%')


def _smooth(pts, n=80):
    """Densify a short polyline so it draws smooth.

    `antialias_line` is the exact routine `hyp.plot(antialias=True)` runs on
    every library-drawn line; we call it directly here because this forecast
    overlay is hand-drawn matplotlib rather than a plotted dataset.
    """
    from hypertools._shared.helpers import antialias_line
    pts = np.asarray(pts, float)
    if len(pts) < 3:
        return pts
    return antialias_line(pts, n)[0]


def _hang(head, delta):
    return _smooth(np.clip(head + delta, BLO, BHI))


# history fan + the current forecast, in the SAME red: these are all
# forecasts, so what separates them is weight and opacity
N_HIST = 16
FC_COLOR = '#E23B2E'
HIST_COLOR = FC_COLOR
hist_lines = [ax.plot([], [], [], '-', color=HIST_COLOR, lw=1.1, alpha=0.0,
                      zorder=6)[0] for _ in range(N_HIST)]
for _ln in hist_lines:
    _ln.set_clip_on(False)
fc_line, = ax.plot([], [], [], '--', color=FC_COLOR, lw=2.6, alpha=0.98,
                   zorder=10)
fc_line.set_clip_on(False)

# labeled colorbar for the equal-weight index
cax = fig.add_axes([0.82, 0.14, 0.02, 0.66])
sm = ScalarMappable(Normalize(idx_level.min(), idx_level.max()), cmap='plasma')
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label('equal-weight index (start = 100)', fontsize=9)

title = fig.text(0.40, 0.965, '', ha='center', va='top', fontsize=14,
                 fontweight='bold', color='#1a1a1a')
# the running score lives UNDER the title, in its own lighter subtitle line
acc_label = fig.text(0.40, 0.925, '', ha='center', va='top', fontsize=11.5,
                     color='#555')
# legend built from REAL Line2D handles, so each entry is drawn in exactly the
# style it has on screen (thick red dashed = live forecast; thin faint red
# solid = the past-forecast fan) instead of two identical text labels
fig.legend(handles=[
    Line2D([], [], color=FC_COLOR, lw=2.6, ls='--',
           label=f'forecast from today (next {HORIZON} months)'),
    Line2D([], [], color=HIST_COLOR, lw=1.1, ls='-', alpha=0.55,
           label='past forecasts, as they were made'),
], loc='lower left', bbox_to_anchor=(0.055, 0.02), ncol=1, frameon=False,
    fontsize=12.5, handlelength=3.2, labelspacing=0.6, labelcolor='#444')
fig.text(0.40, 0.005, 'arrows amplified for visibility; length is relative, '
         'not a price target', ha='center', va='bottom', fontsize=9,
         color='#8a8a8a', style='italic')


def _wrapped(num, *args):
    result = _orig(num, *args)
    hx, hy, hz = market_line.get_data_3d()
    head = np.array([hx[-1], hy[-1], hz[-1]])
    HEAD_CACHE[num] = head
    passed = [f for f in frame_list if f <= num]
    for ln in hist_lines:
        ln.set_alpha(0.0)
    if passed:
        cur = _hang(head, FC[passed[-1]])
        fc_line.set_data(cur[:, 0], cur[:, 1])
        fc_line.set_3d_properties(cur[:, 2])
        prior = passed[:-1][-N_HIST:]
        for slot, f in enumerate(prior):
            hp = _hang(HEAD_CACHE.get(f, head), FC[f])
            hist_lines[slot].set_data(hp[:, 0], hp[:, 1])
            hist_lines[slot].set_3d_properties(hp[:, 2])
            hist_lines[slot].set_alpha(0.08 + 0.30 * (slot + 1) / len(prior))
    else:
        fc_line.set_data([], [])
        fc_line.set_3d_properties([])
    # running directional accuracy of every forecast whose horizon has already
    # elapsed on screen (50% = a coin flip)
    acc = ACC[min(num, total - 1)]
    title.set_text('many markets as one path')
    acc_label.set_text(
        'forecast direction correct so far: — (waiting for first horizon)'
        if np.isnan(acc) else
        f'forecast direction correct so far: {acc:.0f}%   (50% = coin flip)')
    return result


ani._func = _wrapped

forecasts: 74 drawn, 73 scored; final directional accuracy = 66%


## 6. Display the animation

In [7]:
fig.set_dpi(100)  # halve hypertools' default 200-dpi canvas for a lighter GIF
ani.save('market_forecast.gif', fps=fps)
print('saved market_forecast.gif')

saved market_forecast.gif


![market path with chemtrails and live forecast](market_forecast.gif)